# Recover the Claude comparison — reuse your completed Qwen results

**No GPU or Qwen download required.** Upload `larger-model-baseline-results.zip` (or `qwen-7b-baseline-results.zip`), then enter your Claude API key securely. This uses your API account and runs only the missing Claude comparison, on exactly the saved cases. No training or model deployment occurs.

The earlier notebook passed a `temperature` keyword absent from Anthropic SDK 1.15.0; all Claude attempts failed before inference. The corrected helper uses temperature 0 only when the installed SDK exposes it; otherwise sampling uses the provider default. A single setup call must succeed before the remaining batch runs. That first response is reused, not billed twice. The saved Qwen responses are retained unchanged. Claude gets a 512-token output budget to reduce malformed/truncated JSON; the Qwen run used 256 tokens, recorded separately.

The uploaded archive is parsed as data. No embedded scripts or document instructions are executed. The result contains student writing; keep it private. Save the final ZIP and upload it back to the chat for review.


In [ ]:
import pathlib,subprocess,sys,base64,hashlib,json,zipfile
subprocess.run([sys.executable,"-m","pip","install","-q","anthropic>=0.68"],check=True)
root=pathlib.Path("/content/claude-recovery-v2") if pathlib.Path("/content").exists() else pathlib.Path.cwd()/"claude-recovery-v2"
root.mkdir(exist_ok=True)
helper_source=base64.b64decode('IiIiSW5mZXJlbmNlLW9ubHkgYmFzZWxpbmUgZXZhbHVhdGlvbiB3aXRoIG5hdGl2ZSBtZXRyaWNzIGFuZCBmZWVkYmFjayBncm91bmRpbmcgY2hlY2tzLiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgaW5zcGVjdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSBzdGF0aXN0aWNzIGltcG9ydCBtZWFuCmltcG9ydCB0aW1lCmltcG9ydCB1bmljb2RlZGF0YQoKU0NPUkVEX1RSQUNLUz17J2FtcGxpZnlfbmF0aXZlJywnYXNhcDJfaG9saXN0aWMnfQpkZWYgbm9ybWFsaXplKHMpOnJldHVybiAnICcuam9pbih1bmljb2RlZGF0YS5ub3JtYWxpemUoJ05GS0MnLHMpLnNwbGl0KCkpCgpkZWYgcGFyc2VfcmVzcG9uc2UocmF3LGNhc2UpOgogICAgc3RyaWN0PVRydWU7dGV4dD1yYXcuc3RyaXAoKQogICAgaWYgdGV4dC5zdGFydHN3aXRoKCdgYGAnKToKICAgICAgICBzdHJpY3Q9RmFsc2UKICAgICAgICBpZiBub3QgdGV4dC5lbmRzd2l0aCgnYGBgJyk6cmFpc2UgVmFsdWVFcnJvcignSW5jb21wbGV0ZSBmZW5jZWQgSlNPTi4nKQogICAgICAgIHRleHQ9dGV4dC5zcGxpdCgnXG4nLDEpWy0xXS5yc3BsaXQoJ2BgYCcsMSlbMF0uc3RyaXAoKQogICAgdmFsdWU9anNvbi5sb2Fkcyh0ZXh0KQogICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWUsZGljdCk6cmFpc2UgVmFsdWVFcnJvcignRXhwZWN0ZWQgSlNPTiBvYmplY3QuJykKICAgIHJlcXVpcmVkPXsnZmVlZGJhY2snLCdzdHVkZW50X3F1b3RlJ30KICAgIGlmIGNhc2VbJ3RyYWNrJ10gaW4gU0NPUkVEX1RSQUNLUzpyZXF1aXJlZC5hZGQoJ3Njb3JlJykKICAgIGVsaWYgY2FzZVsndHJhY2snXSE9J3VzZXJfc3R5bGVfcHJlZmVyZW5jZSc6cmVxdWlyZWQuYWRkKCduZXh0X3F1ZXN0aW9uJykKICAgIGlmIHNldCh2YWx1ZSkhPXJlcXVpcmVkOnJhaXNlIFZhbHVlRXJyb3IoJ1dyb25nIG91dHB1dCBmaWVsZHMuJykKICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlWydmZWVkYmFjayddLHN0cikgb3IgbGVuKHZhbHVlWydmZWVkYmFjayddLnN0cmlwKCkpPDg6cmFpc2UgVmFsdWVFcnJvcignTWlzc2luZyBmZWVkYmFjay4nKQogICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWVbJ3N0dWRlbnRfcXVvdGUnXSxzdHIpIG9yIG5vdCB2YWx1ZVsnc3R1ZGVudF9xdW90ZSddLnN0cmlwKCk6cmFpc2UgVmFsdWVFcnJvcignTWlzc2luZyBzdHVkZW50IGV4Y2VycHQuJykKICAgIGlmICduZXh0X3F1ZXN0aW9uJyBpbiB2YWx1ZSBhbmQgKG5vdCBpc2luc3RhbmNlKHZhbHVlWyduZXh0X3F1ZXN0aW9uJ10sc3RyKSBvciBsZW4odmFsdWVbJ25leHRfcXVlc3Rpb24nXS5zdHJpcCgpKTw1KTpyYWlzZSBWYWx1ZUVycm9yKCdNaXNzaW5nIHJldmlzaW9uIHF1ZXN0aW9uLicpCiAgICBpZiAnc2NvcmUnIGluIHZhbHVlIGFuZCAodHlwZSh2YWx1ZVsnc2NvcmUnXSkgaXMgbm90IGludCBvciBub3QgY2FzZVsnc2NvcmVfbWluJ108PXZhbHVlWydzY29yZSddPD1jYXNlWydzY29yZV9tYXgnXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW52YWxpZCBuYXRpdmUgc2NvcmUuJykKICAgIHJldHVybiB7Kip2YWx1ZSwnc3RyaWN0X2pzb24nOnN0cmljdCwncXVvdGF0aW9uX3ByZXNlbnQnOm5vcm1hbGl6ZSh2YWx1ZVsnc3R1ZGVudF9xdW90ZSddKSBpbiBub3JtYWxpemUoY2FzZVsnd3JpdGluZyddKX0KCmRlZiBxdWFkcmF0aWNfa2FwcGEocGFpcnMsbG93LGhpZ2gpOgogICAgaWYgbm90IHBhaXJzOnJldHVybiBOb25lCiAgICBuPWxlbihwYWlycyk7ZXhwZWN0ZWQ9MDtvYnNlcnZlZD0wCiAgICBmb3IgYSxiIGluIHBhaXJzOm9ic2VydmVkKz0oYS1iKSoqMgogICAgZm9yIGkgaW4gcmFuZ2UobG93LGhpZ2grMSk6CiAgICAgICAgZm9yIGogaW4gcmFuZ2UobG93LGhpZ2grMSk6CiAgICAgICAgICAgIGV4cGVjdGVkKz0oaS1qKSoqMipzdW0oYT09aSBmb3IgYSxiIGluIHBhaXJzKSpzdW0oYj09aiBmb3IgYSxiIGluIHBhaXJzKS9uCiAgICByZXR1cm4gMS1vYnNlcnZlZC9leHBlY3RlZCBpZiBleHBlY3RlZCBlbHNlIE5vbmUKCmRlZiBtZXRyaWNzKGNhc2VzLGF0dGVtcHRzKToKICAgIHJlZnM9e2NbJ2lkJ106YyBmb3IgYyBpbiBjYXNlc307b2s9W2EgZm9yIGEgaW4gYXR0ZW1wdHMgaWYgYS5nZXQoJ3N0YXR1cycpPT0nb2snXQogICAgc2NvcmVfdHJhY2tzPXt9CiAgICBmb3IgdHJhY2sgaW4gc29ydGVkKFNDT1JFRF9UUkFDS1MpOgogICAgICAgIGVsaWdpYmxlPVtjIGZvciBjIGluIGNhc2VzIGlmIGNbJ3RyYWNrJ109PXRyYWNrXTtpZHM9e2NbJ2lkJ10gZm9yIGMgaW4gZWxpZ2libGV9CiAgICAgICAgc2NvcmVkPVthIGZvciBhIGluIG9rIGlmIGFbJ2lkJ10gaW4gaWRzXQogICAgICAgIGRlZiBibG9jayhzdWJzZXQpOgogICAgICAgICAgICBwYWlycz1bKHJlZnNbYVsnaWQnXV1bJ3JlZmVyZW5jZSddWydzY29yZSddLGFbJ3Njb3JlJ10pIGZvciBhIGluIHN1YnNldF0KICAgICAgICAgICAgcmV0dXJuIHsndmFsaWQnOmxlbihwYWlycyksJ21hZSc6bWVhbihhYnMoYS1iKSBmb3IgYSxiIGluIHBhaXJzKSBpZiBwYWlycyBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICAgICAgJ2V4YWN0X2FncmVlbWVudCc6bWVhbihhPT1iIGZvciBhLGIgaW4gcGFpcnMpIGlmIHBhaXJzIGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgICAgICAnd2l0aGluX29uZSc6bWVhbihhYnMoYS1iKTw9MSBmb3IgYSxiIGluIHBhaXJzKSBpZiBwYWlycyBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICAgICAgJ3NpZ25lZF9iaWFzJzptZWFuKGItYSBmb3IgYSxiIGluIHBhaXJzKSBpZiBwYWlycyBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICAgICAgJ3ByZWRpY3RlZF9zY29yZV9jb3VudHMnOntzdHIodik6c3VtKGI9PXYgZm9yIGEsYiBpbiBwYWlycykgZm9yIHYgaW4gc29ydGVkKHtiIGZvciBhLGIgaW4gcGFpcnN9KX0sCiAgICAgICAgICAgICAgICAgICAgJ3F1YWRyYXRpY193ZWlnaHRlZF9rYXBwYSc6cXVhZHJhdGljX2thcHBhKHBhaXJzLGVsaWdpYmxlWzBdWydzY29yZV9taW4nXSxlbGlnaWJsZVswXVsnc2NvcmVfbWF4J10pIGlmIGVsaWdpYmxlIGVsc2UgTm9uZX0KICAgICAgICBmaXhlZD1bKGNbJ3JlZmVyZW5jZSddWydzY29yZSddLDMpIGZvciBjIGluIGVsaWdpYmxlXQogICAgICAgIHNjb3JlX3RyYWNrc1t0cmFja109eyoqYmxvY2soc2NvcmVkKSwnZXhwZWN0ZWQnOmxlbihlbGlnaWJsZSksJ2ludmFsaWRfb3JfbWlzc2luZyc6bGVuKGVsaWdpYmxlKS1sZW4oc2NvcmVkKSwKICAgICAgICAgICAgJ2V4YWN0X21hdGNoZXNfb3Zlcl9hbGxfY2FzZXMnOnN1bShhWydzY29yZSddPT1yZWZzW2FbJ2lkJ11dWydyZWZlcmVuY2UnXVsnc2NvcmUnXSBmb3IgYSBpbiBzY29yZWQpL2xlbihlbGlnaWJsZSkgaWYgZWxpZ2libGUgZWxzZSBOb25lLAogICAgICAgICAgICAnZml4ZWRfc2NvcmVfM19iYXNlbGluZSc6eydtYWUnOm1lYW4oYWJzKGEtYikgZm9yIGEsYiBpbiBmaXhlZCkgaWYgZml4ZWQgZWxzZSBOb25lLCdleGFjdF9hZ3JlZW1lbnQnOm1lYW4oYT09YiBmb3IgYSxiIGluIGZpeGVkKSBpZiBmaXhlZCBlbHNlIE5vbmV9LAogICAgICAgICAgICAnYnlfcHJvbXB0Jzp7cDpibG9jayhbYSBmb3IgYSBpbiBzY29yZWQgaWYgcmVmc1thWydpZCddXS5nZXQoJ3Byb21wdF9uYW1lJyk9PXBdKSBmb3IgcCBpbiBzb3J0ZWQoe2MuZ2V0KCdwcm9tcHRfbmFtZScsJycpIGZvciBjIGluIGVsaWdpYmxlfSl9IGlmIHRyYWNrPT0nYXNhcDJfaG9saXN0aWMnIGVsc2Uge30sCiAgICAgICAgICAgICdieV9nZW5yZSc6e2c6YmxvY2soW2EgZm9yIGEgaW4gc2NvcmVkIGlmIHJlZnNbYVsnaWQnXV0uZ2V0KCdnZW5yZScsJycpPT1nXSkgZm9yIGcgaW4gc29ydGVkKHtjLmdldCgnZ2VucmUnLCcnKSBmb3IgYyBpbiBlbGlnaWJsZX0pfSB9CiAgICByZXR1cm4geydwbGFubmVkX2Nhc2VzJzpsZW4oY2FzZXMpLCdhdHRlbXB0ZWQnOmxlbihhdHRlbXB0cyksJ3ZhbGlkX291dHB1dHMnOmxlbihvayksJ2ZhaWxlZF9vdXRwdXRzJzpsZW4oYXR0ZW1wdHMpLWxlbihvayksCiAgICAgICAgICAgICdjb21wbGV0ZSc6bGVuKG9rKT09bGVuKGNhc2VzKSwnc3RyaWN0X2pzb25fb3V0cHV0cyc6c3VtKGEuZ2V0KCdzdHJpY3RfanNvbicpIGlzIFRydWUgZm9yIGEgaW4gb2spLAogICAgICAgICAgICAncXVvdGF0aW9uX3ByZXNlbnQnOnN1bShhLmdldCgncXVvdGF0aW9uX3ByZXNlbnQnKSBpcyBUcnVlIGZvciBhIGluIG9rKSwKICAgICAgICAgICAgJ3F1b3RhdGlvbl9hYnNlbnQnOnN1bShhLmdldCgncXVvdGF0aW9uX3ByZXNlbnQnKSBpcyBGYWxzZSBmb3IgYSBpbiBvayksCiAgICAgICAgICAgICdzY29yZV90cmFja3MnOnNjb3JlX3RyYWNrcywndGVhY2hlcl9jYWxpYnJhdGVkJzpGYWxzZSwnZmVlZGJhY2tfcXVhbGl0eSc6J2h1bWFuX3Jldmlld19wZW5kaW5nJywKICAgICAgICAgICAgJ25vdGVzJzpbJ05hdGl2ZSBzY2FsZXMgYXJlIHJlcG9ydGVkIHNlcGFyYXRlbHk7IG5vIGNyb3NzLXJ1YnJpYyBhdmVyYWdlLicsCiAgICAgICAgICAgICAgICAgICAgICdRdW90ZSBwcmVzZW5jZSBpcyBhIG1lY2hhbmljYWwgY2hlY2ssIG5vdCBwcm9vZiBvZiBmYWN0dWFsIG9yIGludGVycHJldGl2ZSBhY2N1cmFjeS4nLAogICAgICAgICAgICAgICAgICAgICAnQWx3YXlzIGluc3BlY3QgaW52YWxpZC1vdXRwdXQgY292ZXJhZ2UgYmVmb3JlIGNvbXBhcmluZyB2YWxpZC1vbmx5IGVycm9yLiddfQoKZGVmIGxvYWRfY2FzZXMoZGlyZWN0b3J5KToKICAgIHJvb3Q9UGF0aChkaXJlY3RvcnkpO209anNvbi5sb2Fkcygocm9vdC8nbWFuaWZlc3QuanNvbicpLnJlYWRfdGV4dCgpKTtkYXRhPShyb290LydjYXNlcy5qc29ubCcpLnJlYWRfYnl0ZXMoKQogICAgaWYgaGFzaGxpYi5zaGEyNTYoZGF0YSkuaGV4ZGlnZXN0KCkhPW1bJ2Nhc2VzX3NoYTI1NiddOnJhaXNlIFZhbHVlRXJyb3IoJ0JlbmNobWFyayBjYXNlcyBjaGFuZ2VkLicpCiAgICByZXR1cm4gW2pzb24ubG9hZHMobGluZSkgZm9yIGxpbmUgaW4gZGF0YS5zcGxpdGxpbmVzKCkgaWYgbGluZS5zdHJpcCgpXQoKZGVmIHJ1bihkaXJlY3Rvcnksb3V0LG1vZGVsX25hbWUsZ2VuZXJhdGUsbWV0YWRhdGEpOgogICAgY2FzZXM9bG9hZF9jYXNlcyhkaXJlY3RvcnkpO291dD1QYXRoKG91dCk7b3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgZmluZ2VycHJpbnQ9aGFzaGxpYi5zaGEyNTYoKFBhdGgoZGlyZWN0b3J5KS8nY2FzZXMuanNvbmwnKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpCiAgICBtZXRhPXsqKm1ldGFkYXRhLCdtb2RlbF9uYW1lJzptb2RlbF9uYW1lLCdjYXNlc19zaGEyNTYnOmZpbmdlcnByaW50LCd0cmFpbmluZ19wZXJmb3JtZWQnOkZhbHNlfQogICAgY29uZmlnX2ZpbGU9b3V0Lyhtb2RlbF9uYW1lKycucnVuLmpzb24nKQogICAgaWYgY29uZmlnX2ZpbGUuZXhpc3RzKCkgYW5kIGpzb24ubG9hZHMoY29uZmlnX2ZpbGUucmVhZF90ZXh0KCkpIT1tZXRhOnJhaXNlIFZhbHVlRXJyb3IoJ1Jlc3VtZSBzZXR0aW5ncyBkaWZmZXI7IHVzZSBhbm90aGVyIG91dHB1dCBkaXJlY3RvcnkuJykKICAgIGNvbmZpZ19maWxlLndyaXRlX3RleHQoanNvbi5kdW1wcyhtZXRhLGluZGVudD0yKSsnXG4nKQogICAgcGF0aD1vdXQvKG1vZGVsX25hbWUrJy5wcmVkaWN0aW9ucy5qc29ubCcpO2F0dGVtcHRzPVtdCiAgICBpZiBwYXRoLmV4aXN0cygpOmF0dGVtcHRzPVtqc29uLmxvYWRzKGxpbmUpIGZvciBsaW5lIGluIHBhdGgucmVhZF90ZXh0KCkuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KICAgIGRvbmU9e2FbJ2lkJ10gZm9yIGEgaW4gYXR0ZW1wdHN9CiAgICBmb3IgaSxjIGluIGVudW1lcmF0ZShjYXNlcywxKToKICAgICAgICBpZiBjWydpZCddIGluIGRvbmU6Y29udGludWUKICAgICAgICBzdGFydD10aW1lLm1vbm90b25pYygpO2E9eydpZCc6Y1snaWQnXSwndHJhY2snOmNbJ3RyYWNrJ10sJ3N0YXR1cyc6J2ZhaWxlZCd9CiAgICAgICAgdHJ5OgogICAgICAgICAgICByYXc9Z2VuZXJhdGUoY1snaW5wdXRfbWVzc2FnZXMnXSk7YVsncmF3X3Jlc3BvbnNlJ109cmF3CiAgICAgICAgICAgIGEudXBkYXRlKHBhcnNlX3Jlc3BvbnNlKHJhdyxjKSk7YVsnc3RhdHVzJ109J29rJwogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICAjIFJldGFpbiBjbGFzcyBvbmx5OyBuZXR3b3JrIGVycm9yIHN0cmluZ3MgbWF5IGNvbnRhaW4gc2VjcmV0cyBvciBwcm92aWRlciBib2RpZXMuCiAgICAgICAgICAgIGFbJ2Vycm9yX3R5cGUnXT10eXBlKGV4YykuX19uYW1lX18KICAgICAgICBhWydzZWNvbmRzJ109cm91bmQodGltZS5tb25vdG9uaWMoKS1zdGFydCwzKTthdHRlbXB0cy5hcHBlbmQoYSkKICAgICAgICB3aXRoIHBhdGgub3BlbignYScpIGFzIGY6Zi53cml0ZShqc29uLmR1bXBzKGEsZW5zdXJlX2FzY2lpPUZhbHNlKSsnXG4nKQogICAgICAgIGlmIGklMTA9PTAgb3IgaT09bGVuKGNhc2VzKTpwcmludChtb2RlbF9uYW1lLGksJy8nLGxlbihjYXNlcyksJ2Nhc2VzIHByb2Nlc3NlZCcpCiAgICByZXN1bHQ9bWV0cmljcyhjYXNlcyxhdHRlbXB0cyk7KG91dC8obW9kZWxfbmFtZSsnLm1ldHJpY3MuanNvbicpKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LGluZGVudD0yKSsnXG4nKQogICAgcmV0dXJuIHJlc3VsdAoKZGVmIHdyaXRlX3JldmlldyhkaXJlY3Rvcnksb3V0KToKICAgIGNhc2VzPWxvYWRfY2FzZXMoZGlyZWN0b3J5KTtvdXQ9UGF0aChvdXQpCiAgICBwcmVkaWN0aW9ucz17cC5uYW1lLnNwbGl0KCcucHJlZGljdGlvbnMuanNvbmwnKVswXTp7clsnaWQnXTpyIGZvciByIGluIChqc29uLmxvYWRzKHgpIGZvciB4IGluIHAucmVhZF90ZXh0KCkuc3BsaXRsaW5lcygpIGlmIHguc3RyaXAoKSl9IGZvciBwIGluIG91dC5nbG9iKCcqLnByZWRpY3Rpb25zLmpzb25sJyl9CiAgICByb3dzPVt7KipjLCdtb2RlbF9vdXRwdXRzJzp7bW9kZWw6cm93cy5nZXQoY1snaWQnXSkgZm9yIG1vZGVsLHJvd3MgaW4gcHJlZGljdGlvbnMuaXRlbXMoKX0sCiAgICAgICAgICAgJ2h1bWFuX3Jldmlldyc6eydhY2N1cmFjeSc6Tm9uZSwndXNlZnVsbmVzcyc6Tm9uZSwndW5zdXBwb3J0ZWRfY2xhaW1zJzpOb25lLCdwcmVmZXJyZWRfbW9kZWwnOk5vbmV9fSBmb3IgYyBpbiBjYXNlc10KICAgIChvdXQvJ2h1bWFuLXJldmlldy5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKHJvd3MsZW5zdXJlX2FzY2lpPUZhbHNlLGluZGVudD0yKSsnXG4nKQogICAgcGFpcmVkPXt9CiAgICBuYW1lcz1zb3J0ZWQocHJlZGljdGlvbnMpCiAgICBmb3IgaSxhIGluIGVudW1lcmF0ZShuYW1lcyk6CiAgICAgICAgZm9yIGIgaW4gbmFtZXNbaSsxOl06CiAgICAgICAgICAgIHZhbHVlcz1bXQogICAgICAgICAgICBmb3IgYyBpbiBjYXNlczoKICAgICAgICAgICAgICAgIGlmIGNbJ3RyYWNrJ10gbm90IGluIFNDT1JFRF9UUkFDS1M6Y29udGludWUKICAgICAgICAgICAgICAgIHg9cHJlZGljdGlvbnNbYV0uZ2V0KGNbJ2lkJ10se30pO3k9cHJlZGljdGlvbnNbYl0uZ2V0KGNbJ2lkJ10se30pCiAgICAgICAgICAgICAgICBpZiB4LmdldCgnc3RhdHVzJyk9PSdvaycgYW5kIHkuZ2V0KCdzdGF0dXMnKT09J29rJzoKICAgICAgICAgICAgICAgICAgICBnb2xkPWNbJ3JlZmVyZW5jZSddWydzY29yZSddO3ZhbHVlcy5hcHBlbmQoKGNbJ3RyYWNrJ10sYWJzKHhbJ3Njb3JlJ10tZ29sZCksYWJzKHlbJ3Njb3JlJ10tZ29sZCkpKQogICAgICAgICAgICBwYWlyZWRbYSsnX3ZzXycrYl09e3RyYWNrOnsncGFpcmVkX2Nhc2VzJzpzdW0odD09dHJhY2sgZm9yIHQseCx5IGluIHZhbHVlcyksCiAgICAgICAgICAgICAgICAnYV9sb3dlcl9lcnJvcic6c3VtKHQ9PXRyYWNrIGFuZCB4PHkgZm9yIHQseCx5IGluIHZhbHVlcyksJ2JfbG93ZXJfZXJyb3InOnN1bSh0PT10cmFjayBhbmQgeTx4IGZvciB0LHgseSBpbiB2YWx1ZXMpLAogICAgICAgICAgICAgICAgJ2VxdWFsX2Vycm9yJzpzdW0odD09dHJhY2sgYW5kIHg9PXkgZm9yIHQseCx5IGluIHZhbHVlcyl9IGZvciB0cmFjayBpbiBzb3J0ZWQoU0NPUkVEX1RSQUNLUyl9CiAgICAob3V0LydwYWlyZWQtY29tcGFyaXNvbi5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKHBhaXJlZCxpbmRlbnQ9MikrJ1xuJykKCgpkZWYgY2xhdWRlX3RleHQoY2xpZW50LG1vZGVsLG1lc3NhZ2VzLG1heF90b2tlbnM9NTEyKToKICAgICIiIkNhbGwgdGhlIGluc3RhbGxlZCBTREsgd2l0aG91dCBwYXNzaW5nIHVuc3VwcG9ydGVkIHNhbXBsaW5nIGFyZ3VtZW50cy4iIiIKICAgIGt3YXJncz17J21vZGVsJzptb2RlbCwnbWF4X3Rva2Vucyc6bWF4X3Rva2Vucywnc3lzdGVtJzptZXNzYWdlc1swXVsnY29udGVudCddLCdtZXNzYWdlcyc6bWVzc2FnZXNbMTpdfQogICAgaWYgJ3RlbXBlcmF0dXJlJyBpbiBpbnNwZWN0LnNpZ25hdHVyZShjbGllbnQubWVzc2FnZXMuY3JlYXRlKS5wYXJhbWV0ZXJzOgogICAgICAgIGt3YXJnc1sndGVtcGVyYXR1cmUnXT0wCiAgICByZXN1bHQ9Y2xpZW50Lm1lc3NhZ2VzLmNyZWF0ZSgqKmt3YXJncykKICAgIHJldHVybiAnJy5qb2luKGJsb2NrLnRleHQgZm9yIGJsb2NrIGluIHJlc3VsdC5jb250ZW50IGlmIGdldGF0dHIoYmxvY2ssJ3R5cGUnLE5vbmUpPT0ndGV4dCcpCg==')
helper_sha256='383c93bc2bbd32080095be968cb5f007fb779848e8714c50a31080922fba9a68'
assert hashlib.sha256(helper_source).hexdigest()==helper_sha256
(root/"baseline_runner.py").write_bytes(helper_source)
sys.path.insert(0,str(root))
from baseline_runner import load_cases,run,write_review,metrics,claude_text
print("Corrected Claude helper ready. No GPU needed.")


In [ ]:
from google.colab import files
uploaded=files.upload()
archives=[pathlib.Path(name) for name in uploaded if name.endswith(".zip")]
if len(archives)!=1:raise RuntimeError("Upload one completed baseline-results ZIP.")
results=root/"results";results.mkdir(exist_ok=True)
dataset=root/"dataset";dataset.mkdir(exist_ok=True)
allowed={"QWEN_LICENSE.txt","ATTRIBUTION.txt","README.txt","runtime-versions.json","qwen-model.json", "asap2-provenance.json","foxglove-provenance.json","source-extraction-methods.json","manifest.json","qwen_7b_baseline.run.json","qwen_7b_baseline.predictions.jsonl","qwen_7b_baseline.metrics.json","code-snapshot.json"}
with zipfile.ZipFile(archives[0]) as archive:
    if sum(x.file_size for x in archive.infolist())>64*1024*1024:raise RuntimeError("Unexpectedly large result archive.")
    review=json.loads(archive.read("human-review.json"))
    manifest=json.loads(archive.read("manifest.json"))
    cases=[{k:v for k,v in row.items() if k not in ("model_outputs","human_review")} for row in review]
    case_bytes=''.join(json.dumps(c,ensure_ascii=False)+'\n' for c in cases).encode()
    if hashlib.sha256(case_bytes).hexdigest()!=manifest["cases_sha256"]:raise RuntimeError("Reconstructed case checksum differs from the evaluated dataset.")
    (dataset/"cases.jsonl").write_bytes(case_bytes)
    (dataset/"manifest.json").write_text(json.dumps(manifest,indent=2))
    for filename in allowed:
        if filename in archive.namelist():(results/filename).write_bytes(archive.read(filename))
# Preserve raw Qwen predictions, and recompute summary using corrected grouping.
cases=load_cases(dataset)
qwen=[json.loads(line) for line in (results/"qwen_7b_baseline.predictions.jsonl").read_text().splitlines() if line.strip()]
(results/"qwen_7b_baseline.metrics.json").write_text(json.dumps(metrics(cases,qwen),indent=2))
print("Verified",len(cases),"original cases. Qwen predictions preserved; no Qwen rerun.")


In [ ]:
from getpass import getpass
import anthropic,inspect
CLAUDE_MODEL="claude-haiku-4-5"
claude_key=getpass("Anthropic API key: ").strip()
if not claude_key:raise RuntimeError("A Claude key is required for this comparison; no calls made.")
client=anthropic.Anthropic(api_key=claude_key,timeout=120,max_retries=2)
meta={"model":CLAUDE_MODEL,"max_new_tokens":512,
      "temperature_control":"zero" if "temperature" in inspect.signature(client.messages.create).parameters else "provider_default",
      "anthropic_sdk_version":anthropic.__version__}
try:
    first_messages=cases[0]["input_messages"]
    try:
        first_response=claude_text(client,CLAUDE_MODEL,first_messages,max_tokens=512)
    except Exception as exc:
        status=getattr(exc,"status_code",None)
        raise RuntimeError("Claude setup failed: "+type(exc).__name__+(" HTTP "+str(status) if status else "")+". The batch was not started. Check API key/account/model access.") from None
    cache={json.dumps(first_messages,sort_keys=True):first_response}
    def generate(messages):
        key=json.dumps(messages,sort_keys=True)
        return cache.pop(key) if key in cache else claude_text(client,CLAUDE_MODEL,messages,max_tokens=512)
    result=run(dataset,results,"claude_baseline",generate,meta)
    print(json.dumps(result,indent=2))
finally:
    client.close();claude_key=""


In [ ]:
import shutil
write_review(dataset,results)
(results/"recovery-note.json").write_text(json.dumps({"helper_sha256":helper_sha256,
    "qwen_recomputed":False,"claude_sdk_temperature_compatibility_fixed":True,
    "claude_max_tokens":512,"qwen_max_tokens":256,"training_performed":False},indent=2))
archive=shutil.make_archive(str(root/"recovered-model-comparison"),"zip",results)
files.download(archive)
print("Upload recovered-model-comparison.zip back to the chat.")
